<a href="https://colab.research.google.com/github/jpmayery/Colab-Astro/blob/main/notebooks/run_triangles_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Run Triangles — Exécution Colab
Ce notebook exécute le script `run_triangles` en Colab.
- Cellule 2 définit la fonction `run_triangles` et les paramètres.
- Cellule 3 exécute le run canonique `MIN_NEUTRON=29, MIN_GLOBAL=29`.
Modifiez les paramètres et relancez les cellules si besoin (Runtime → Run all).

In [ ]:
import math
import statistics as stats
from pprint import pprint

PETITS = {5,7,11,13}
LAMBDA_PHYS = 1836.15267389

def coherent_gcd(m, n, k):
    return math.gcd(m,n) == math.gcd(n,k) == math.gcd(m,k)

def heron_area(m, n, k):
    s = 0.5*(m+n+k)
    val = s*(s-m)*(s-n)*(s-k)
    if val <= 0:
        return 0.0
    return math.sqrt(val)

def Q_T(m,n,k):
    d1 = n-m
    d2 = k-n
    diff = d2-d1
    Q = 0
    if diff>0: Q=1
    elif diff<0: Q=-1
    T = abs(diff)
    return Q, T

def mass_geo(m,n,k):
    _, T = Q_T(m,n,k)
    A = heron_area(m,n,k)
    return A/(1.0+T*T)

def aspect(m,n,k):
    return k/ m

def count_petits(m,n,k):
    return int(m in PETITS)+int(n in PETITS)+int(k in PETITS)

def has_any_petit(m,n,k):
    return (m in PETITS) or (n in PETITS) or (k in PETITS)

def run_triangles(N=60, MIN_NEUTRON=None, MIN_GLOBAL=None,
                  T_NEUTRON_MAX=1, ASPECT_NEUTRON_MAX=2, ASPECT_PROTON_MAX=3, ASPECT_ELECTRON_MIN=4):
    Cn, Cp, Ce = [], [], []
    for m in range(1, N+1):
        for n in range(m+1, N+1):
            for k in range(n+1, N+1):
                if m+n <= k:
                    continue
                if MIN_GLOBAL is not None and min(m,n,k) < MIN_GLOBAL:
                    continue
                if MIN_NEUTRON is not None and min(m,n,k) < MIN_NEUTRON:
                    pass_neutron_min = False
                else:
                    pass_neutron_min = True

                # neutron-like
                if pass_neutron_min and coherent_gcd(m,n,k) and not has_any_petit(m,n,k):
                    Q,T = Q_T(m,n,k)
                    if T <= T_NEUTRON_MAX and aspect(m,n,k) <= ASPECT_NEUTRON_MAX:
                        Cn.append((m,n,k))
                # proton-like
                if coherent_gcd(m,n,k) and count_petits(m,n,k)==1:
                    Q,T = Q_T(m,n,k)
                    if Q==1 and aspect(m,n,k) <= ASPECT_PROTON_MAX:
                        Cp.append((m,n,k))
                # electron-like
                if coherent_gcd(m,n,k) and count_petits(m,n,k)==1:
                    Q,T = Q_T(m,n,k)
                    if Q==-1 and aspect(m,n,k) >= ASPECT_ELECTRON_MIN:
                        Ce.append((m,n,k))

    Mn_list = [mass_geo(*t) for t in Cn]
    Mp_list = [mass_geo(*t) for t in Cp]
    Me_list = [mass_geo(*t) for t in Ce]

    def mean(x): return sum(x)/len(x) if x else float('nan')
    def median(x): return stats.median(x) if x else float('nan')

    Mn = mean(Mn_list); Mp = mean(Mp_list); Me = mean(Me_list)
    medMn = median(Mn_list)
    Cint_n = Mn/medMn if (medMn and not math.isnan(Mn)) else float('nan')
    lambda_geo = Mp/Me if (Me and not math.isnan(Mp) and not math.isnan(Me)) else float('nan')
    lambda_pred = 1728.0 * Cint_n if not math.isnan(Cint_n) else float('nan')
    eps = (LAMBDA_PHYS - lambda_pred)/LAMBDA_PHYS if not math.isnan(lambda_pred) else float('nan')

    out = {
        'N': N, 'MIN_NEUTRON': MIN_NEUTRON, 'MIN_GLOBAL': MIN_GLOBAL,
        '|Cn|': len(Cn), '|Cp|': len(Cp), '|Ce|': len(Ce),
        'Mn': Mn, 'Mp': Mp, 'Me': Me, 'median_Mn': medMn, 'Cint_n': Cint_n,
        'lambda_geo': lambda_geo, 'lambda_pred': lambda_pred, 'eps': eps,
    }
    return out

In [ ]:
# Run canonique demandé : MIN_NEUTRON=29, MIN_GLOBAL=29
res = run_triangles(N=60, MIN_NEUTRON=29, MIN_GLOBAL=29)
from pprint import pprint
pprint(res)

## Instructions
- Exécutez `Runtime -> Run all` pour lancer le calcul en Colab.
- Vous pouvez modifier la cellule d'exécution (paramètres) pour tester d'autres seuils (None, 23, 29).

In [3]:
# Runs comparatifs et génération LaTeX
def run_cases():
    cases = [
        ('sans_seuil', dict(N=60, MIN_NEUTRON=None, MIN_GLOBAL=None)),
        ('seuil_23', dict(N=60, MIN_NEUTRON=23, MIN_GLOBAL=None)),
        ('seuil_29', dict(N=60, MIN_NEUTRON=29, MIN_GLOBAL=None)),
        ('global_29_N200', dict(N=200, MIN_NEUTRON=29, MIN_GLOBAL=29)),
    ]
    results = {}
    for name, params in cases:
        print(f'Running {name} with', params)
        res = run_triangles(**params)
        results[name] = res
        print(res)
        print('
---
')

    # Sweep MIN_NEUTRON values to observe stability (N=200, no MIN_GLOBAL)
    sweep = []
    for mn in range(20, 36):
        r = run_triangles(N=200, MIN_NEUTRON=mn, MIN_GLOBAL=None)
        sweep.append((mn, r['Cint_n'], r['lambda_pred'], r['|Cn|']))
    results['sweep_20_35_N200'] = sweep

    # Build LaTeX table for the four main cases
    def fmt(x): return f"{x:.6f}" if isinstance(x, float) and not math.isnan(x) else ('nan' if isinstance(x, float) else str(x))
    tex = []
    tex.append('\\begin{tabular}{lrrrr}')
    tex.append('\\toprule')
    tex.append('Run & |\\mathcal{C}_n| & C_{\\mathrm{int}}^{(n)} & \\ambda_{\\mathrm{pred}} & \\arepsilon\\\\')
    tex.append('\\midrule')
    for key in ['sans_seuil','seuil_23','seuil_29','global_29_N200']:
        r = results[key]
        tex.append(f"{key} & {r['|Cn|']} & {fmt(r['Cint_n'])} & {fmt(r['lambda_pred'])} & {fmt(r['eps'])}\\")
    tex.append('\\bottomrule')
    tex.append('\\end{tabular}')

    results['latex_table'] = '\n'.join(tex)
    return results

res_all = run_cases()
print('LaTeX table for annex (copy-paste):
')
print(res_all['latex_table'])

SyntaxError: unterminated string literal (detected at line 15) (3638057896.py, line 15)

In [ ]:
from pathlib import Path
import json
import runpy


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "python" / "scripts" / "neutral_atom_check.py").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root")


root = find_project_root()
scripts = root / "python" / "scripts"

neutral_ns = runpy.run_path(str(scripts / "neutral_atom_check.py"))
rust_ns = runpy.run_path(str(scripts / "iron_rust_check.py"))

neutral_result = neutral_ns["run_check"]()
rust_result = rust_ns["run_check"]()

summary = {
    "neutrality": {
        "verdict": neutral_result["verdict"],
        "json_path": neutral_result["json_path"],
        "report_path": neutral_result["report_path"],
    },
    "rust": {
        "verdict": rust_result["verdict"],
        "json_path": rust_result["json_path"],
        "report_path": rust_result["report_path"],
    },
}

print(json.dumps(summary, indent=2, ensure_ascii=False))